# 04 — NIAH retention curves and the control battery

**Stage:** Proposal Stages 3–4 on a single checkpoint. **Produces:** Table 4 (controls
C1–C4), Table 6 (retention summary), and the raw rows behind Figures 4–7.

### What changed from the pilot

| pilot | here | why |
|---|---|---|
| decoded `ahn_raw` (pre-`o_proj`) | decodes `o_t = o_proj(ahn_raw)` | the pilot's vector was in head-concat space; dims coincide at 3B so it ran and returned noise |
| `vec @ unembed.T` | `readout_logits` with final RMSNorm | Qwen applies `model.norm` before `lm_head` |
| C1 as `o_t(AHN) − o_t(NOWRITE)` ≡ `o_t` | C1 on the **residual stream** | the AHN output is zero under NOWRITE by construction, so the control was vacuous |
| needle at token ~5 | needle placed past `num_attn_sinks` | tokens in the sink prefix are never compressed |
| best layer chosen on the same data it is plotted from | layers fixed in advance | selection-on-test |
| 2 needles (a third silently dropped) | needles filtered up front | cohort size becomes a decision |
| no CIs, no fit diagnostics | bootstrap CIs and R² | Table 6's R² column decides whether "half-life" is even meaningful |

**Prerequisite:** notebook 01 gates pass and `02_table3_jlens_validation.json` says
`TABLE_3_PASSED: true`. If the lens is not validated, run this with `USE_JLENS=False`
to get logit-lens numbers and label every figure "logit lens, preliminary" — that is a
legitimate pilot, but it is not RQ2.


In [1]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-23fd/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /home/jupyter-dphs-23fd/Interpretability-study-of-Artificial-Hippocampus-Networks


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
EXP = dict(
    layers              = [9, 18, 27],          # fixed in advance, matches the J-lens map
    # Prompt length is roughly num_attn_sinks + sliding_window + eviction_distance, so
    # each distance sets the cost of its own conditions: 8192 -> ~16.4K tokens, 16384 ->
    # ~24.6K. Those two dominated the sweep budget. 16384 is dropped from run 1 and added
    # back only if the decay curve has not flattened by 8192 -- six points still support
    # the exponential fit, and Table 6's R2 column is what says whether it does.
    #
    # distance=0 is also dropped: build_niah_prompt puts the needle at ~145 and the
    # compression boundary at n - sliding_window, which for distance=0 lands at ~146, so
    # the ACTUAL eviction distance is ~1 token and the needle_is_evicted check
    # (sinks <= needle_pos < window_start) is one token from failing. Some filler
    # variants would be silently dropped. 64 is the smallest distance that is safely
    # past the boundary for every filler.
    eviction_distances  = [64, 256, 512, 1024, 2048, 4096, 8192],   # add 16384 if needed
    needle_candidates   = ["Paris", "banana", "Tokyo", "violin", "cinnamon",
                           "harbour", "lantern", "sapphire", "meadow", "trumpet"],
    n_filler_variants   = 3,                    # repeats per (needle, distance)
    use_jlens           = True,
    jlens_path          = os.path.join(CFG["results_dir"], "jlens_qwen25_3b.pt"),
)
print(json.dumps(EXP, indent=2))


{
  "layers": [
    9,
    18,
    27
  ],
  "eviction_distances": [
    64,
    256,
    512,
    1024,
    2048,
    4096,
    8192
  ],
  "needle_candidates": [
    "Paris",
    "banana",
    "Tokyo",
    "violin",
    "cinnamon",
    "harbour",
    "lantern",
    "sapphire",
    "meadow",
    "trumpet"
  ],
  "n_filler_variants": 3,
  "use_jlens": true,
  "jlens_path": "results/run_3b_gdn/jlens_qwen25_3b.pt"
}


In [4]:
CFG["model_path"] = "/home/jupyter-dphs-23fd/Interpretability-study-of-Artificial-Hippocampus-Networks/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN"

In [5]:
import torch, numpy as np, time
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)

# Loading the lens and checking Table 3 are two separate questions and used to share a
# try/except. That was a hard blocker: TABLE_3_PASSED is currently False (checks 2 and 3
# fail, see notebook 02), the `except` caught FileNotFoundError only, so the AssertionError
# escaped and killed the notebook here -- before a single measurement -- even though
# README "Next steps" item 4 explicitly says to run this WITH the J-lens.
#
# Now: a missing .pt falls back to the logit lens (unchanged behaviour), a missing or
# failing Table 3 is a loud warning that stamps lens_validated=False onto every saved row.
lens = None
LENS_VALIDATED = False

if EXP["use_jlens"]:
    try:
        lens = ai.JacobianLens.load(EXP["jlens_path"], map_location=str(bundle.model.device))
        print("J-lens loaded, layers:", sorted(lens.jacobians))
    except FileNotFoundError:
        print("! no J-lens found at", EXP["jlens_path"])
        print("  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.")
        print("  This is not RQ2.")
        EXP["use_jlens"] = False

if EXP["use_jlens"]:
    try:
        v = ai.load_json("02_table3_jlens_validation.json")
        LENS_VALIDATED = bool(v.get("TABLE_3_PASSED"))
        print("Table 3 passed:", LENS_VALIDATED)
    except FileNotFoundError:
        print("! 02_table3_jlens_validation.json not found in", CFG["results_dir"])
        print("  It was produced on the GPU box by notebook 02 but never downloaded.")
        print("  Proceeding with lens_validated=False.")

    if not LENS_VALIDATED:
        print()
        print("!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.")
        print("   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known")
        print("   facts. It does beat the plain logit lens by 8-204x on the same prompts,")
        print("   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.")
        print("   This notebook's control battery (C1/C2/C4) tests that property directly,")
        print("   which is exactly the evidence Gautam asked for before ruling on the lens.")
        print("   Every row is stamped lens_validated=False; label every figure")
        print("   'J-lens, not validated on Table 3' until that ruling lands.")

READOUT = "jlens" if EXP["use_jlens"] else "logit_lens"
EXP["lens_validated"] = LENS_VALIDATED
print()
print("readout:", READOUT, "| lens_validated:", LENS_VALIDATED)


/home/jupyter-dphs-23fd/ahn-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.57it/s]

J-lens loaded, layers: [9, 18, 27]
Table 3 passed: False

!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.
   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known
   facts. It does beat the plain logit lens by 8-204x on the same prompts,
   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.
   This notebook's control battery (C1/C2/C4) tests that property directly,
   which is exactly the evidence Gautam asked for before ruling on the lens.
   Every row is stamped lens_validated=False; label every figure
   'J-lens, not validated on Table 3' until that ruling lands.

readout: jlens | lens_validated: False


In [6]:
needles = ai.single_token_needles(tok, EXP["needle_candidates"])
assert len(needles) >= 5, "need at least 5 single-token needles for a usable cohort"

# distractors for control C2: semantically near the needle, absent from the context
DISTRACTORS = {"Paris": "London", "banana": "mango", "Tokyo": "Osaka",
               "violin": "cello", "cinnamon": "nutmeg", "harbour": "wharf",
               "lantern": "torch", "sapphire": "emerald", "meadow": "pasture",
               "trumpet": "clarinet"}
distractor_ids = {}
for n in needles:
    d = DISTRACTORS.get(n)
    ids = tok.encode(f" {d}", add_special_tokens=False) if d else []
    if len(ids) == 1:
        distractor_ids[n] = ids[0]
print(f"{len(distractor_ids)}/{len(needles)} needles have a single-token distractor")


dropped multi-token needles: {'sapphire': 2, 'meadow': 2}
kept 8 single-token needles: ['Paris', 'Tokyo', 'banana', 'cinnamon', 'harbour', 'lantern', 'trumpet', 'violin']
4/8 needles have a single-token distractor


## The measurement

One row per `(needle, eviction distance, filler variant, layer)`. Each row carries
everything Tables 4, 6 and 8 need, plus the four controls, so the whole battery comes
out of one sweep rather than four.


In [7]:
@torch.no_grad()
def measure(needle, needle_id, distance, filler_idx, in_window=False, shuffle=False):
    spec = ai.build_niah_prompt(tok, needle, bundle, eviction_distance=distance,
                                in_window=in_window, filler_idx=filler_idx)
    if not spec["ahn_will_activate"]:
        return []
    if not in_window and not spec["needle_is_evicted"]:
        return []

    prompt = spec["prompt"]
    if shuffle:   # control C3 — destroy word order, keep the token multiset
        w = prompt.split()
        np.random.default_rng(ai.SEED).shuffle(w)
        prompt = " ".join(w)

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
    off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        # C1: residual-stream difference, the non-vacuous zero-state control
        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        row = {
            "needle": needle, "layer": L, "readout": READOUT,
            # travels with the data so the caveat cannot be lost between here and
            # a figure caption -- see the lens block above
            "lens_validated": LENS_VALIDATED,
            "requested_distance": distance,
            "eviction_distance": spec["actual_eviction_distance"],
            "in_window": in_window, "shuffled": shuffle, "filler_idx": filler_idx,
            "n_tokens": spec["n_tokens"], "needle_pos": spec["needle_pos"],
            "rank": ai.token_rank(lg, needle_id),
            "p_mem": ai.token_prob(lg, needle_id),
            "entropy": ai.readout_entropy(lg),
            "o_t_norm": float(o_t.float().norm()),
            "rank_c1_residual": ai.token_rank(lg_c1, needle_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, needle_id),
        }
        if needle in distractor_ids:      # C2
            row["rank_distractor"] = ai.token_rank(lg, distractor_ids[needle])
            row["p_distractor"] = ai.token_prob(lg, distractor_ids[needle])
        out.append(row)
    return out


In [8]:
rows, t0 = [], time.time()
total = len(needles) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for needle, nid in needles.items():
    for dist in EXP["eviction_distances"]:
        for fi in range(EXP["n_filler_variants"]):
            rows += measure(needle, nid, dist, fi)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()
print(f"main sweep: {len(rows)} rows in {(time.time()-t0)/60:.1f} min")


OutOfMemoryError: CUDA out of memory. Tried to allocate 246.00 MiB. GPU 0 has a total capacity of 19.62 GiB of which 72.94 MiB is free. Process 2211615 has 13.02 GiB memory in use. Including non-PyTorch memory, this process has 6.51 GiB memory in use. Of the allocated memory 6.09 GiB is allocated by PyTorch, and 220.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# C4 pre-eviction baseline (the ceiling) and C3 shuffled context
ctrl_rows = []
for needle, nid in needles.items():
    for fi in range(EXP["n_filler_variants"]):
        ctrl_rows += measure(needle, nid, 0, fi, in_window=True)          # C4 ceiling
    for dist in (1024, 4096):
        ctrl_rows += measure(needle, nid, dist, 0, shuffle=True)          # C3
    ai.free_cuda()
print(f"control rows: {len(ctrl_rows)}")

all_rows = rows + ctrl_rows
ai.save_json({"rows": all_rows, "cfg": CFG, "exp": EXP,
              "needles": needles, "distractors": distractor_ids},
             "04_retention_rows.json")
print("saved -> 04_retention_rows.json  (this is the file notebook 05 reads)")


## Table 4 — the control battery, evaluated

C1 and C4 have to pass before Table 6 is filled in. C3 failing is *not* a bug — the
Expected-Tables document flags it as potentially the most publishable result in the
project: if shuffling the context barely changes retention, AHN is closer to a learned
recency mechanism than to content memory, which contradicts the framing of the original
AHN paper.


In [ ]:
import numpy as np
V = bundle.vocab
def sel(**kw):
    out = all_rows
    for k, v in kw.items():
        out = [r for r in out if r.get(k) == v]
    return out

main   = [r for r in all_rows if not r["in_window"] and not r["shuffled"]]
inwin  = [r for r in all_rows if r["in_window"]]
shuf   = [r for r in all_rows if r["shuffled"]]

T4 = {}

# C1 — the memory's contribution must beat what the residual difference alone explains,
#      and both must beat chance.
T4["C1_zero_state"] = {
    "mean_rank_o_t": float(np.mean([r["rank"] for r in main])),
    "mean_rank_residual_delta": float(np.mean([r["rank_c1_residual"] for r in main])),
    "chance_rank": V / 2,
    "passed": bool(np.mean([r["rank"] for r in main]) < V / 10),
    "note": "fails if the target is at chance in the memory readout: nothing is retained, "
            "or the readout is still in the wrong basis",
}

# C2 — the true needle must beat a semantically near absent token by >= 1 order of magnitude
withd = [r for r in main if "p_distractor" in r]
if withd:
    ratio = float(np.median([(r["p_mem"] + 1e-12) / (r["p_distractor"] + 1e-12) for r in withd]))
    T4["C2_distractor"] = {"median_prob_ratio": ratio, "n": len(withd),
                           "passed": bool(ratio >= 10.0),
                           "note": "below 10x: the readout reflects topic, not the stored item; "
                                   "RQ2 weakens to 'semantic gist'"}

# C3 — shuffling should hurt retention if the state stores content rather than recency
if shuf:
    T4["C3_shuffled_context"] = {
        "mean_rank_ordered": float(np.mean([r["rank"] for r in main
                                            if r["requested_distance"] in (1024, 4096)])),
        "mean_rank_shuffled": float(np.mean([r["rank"] for r in shuf])),
        "passed": bool(np.mean([r["rank"] for r in shuf])
                       > np.mean([r["rank"] for r in main
                                  if r["requested_distance"] in (1024, 4096)])),
        "note": "FAILURE HERE IS A FINDING, not a bug — see Expected_Tables_and_Figures §3",
    }

# C4 — pre-eviction ceiling must be BETTER than any evicted condition.
#      In the pilot it was worse (Paris baseline rank 110712 vs ~95000 evicted), which
#      is the single clearest sign the measurement was not measuring retention.
if inwin:
    T4["C4_pre_eviction_baseline"] = {
        "mean_rank_in_window": float(np.mean([r["rank"] for r in inwin])),
        "mean_rank_evicted": float(np.mean([r["rank"] for r in main])),
        "passed": bool(np.mean([r["rank"] for r in inwin])
                       < np.mean([r["rank"] for r in main])),
        "note": "if the in-window ceiling is worse than the evicted condition, the "
                "placement or the readout is wrong. Stop and fix before Table 6.",
    }

T4["BATTERY_PASSED"] = bool(T4["C1_zero_state"]["passed"]
                            and T4.get("C4_pre_eviction_baseline", {}).get("passed", True))
ai.save_json(T4, "04_table4_controls.json")
print(json.dumps(T4, indent=2))
print("\nC1+C4:", "PASS -> Table 6 may be populated" if T4["BATTERY_PASSED"]
      else "FAIL -> fix instrumentation, do NOT report Table 6")


## Adding more metrics

In [ ]:
import json, numpy as np
data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]

for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    print(f"layer {L}: mean_rank={np.mean([r['rank'] for r in rs]):.0f}  "
          f"mean_p_mem={np.mean([r['p_mem'] for r in rs]):.3e}")

# raw p_mem / p_distractor pairs, unrounded, no epsilon
withd = [r for r in main if "p_distractor" in r][:10]
for r in withd:
    print(r["needle"], r["layer"], r["eviction_distance"],
          "p_mem=", r["p_mem"], "p_distractor=", r["p_distractor"])

In [ ]:
import json, numpy as np

data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
V = 151936
EPS = 1e-30  # small enough not to swamp probabilities down to ~1e-24

print("=== C1 per layer (pass bar: mean_rank < %d) ===" % (V // 10))
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    mr = np.mean([r["rank"] for r in rs])
    print(f"layer {L}: n={len(rs)} mean_rank={mr:.0f}  passes={mr < V/10}")

print("\n=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===")
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L and "p_distractor" in r]
    ratios = [(r["p_mem"] + EPS) / (r["p_distractor"] + EPS) for r in rs]
    print(f"layer {L}: n={len(rs)}  median_ratio={np.median(ratios):.3f}  "
          f"frac_needle>distractor={np.mean([r>1 for r in ratios]):.2f}  "
          f"frac_pass_10x={np.mean([r>=10 for r in ratios]):.2f}")

### Gate for this notebook

- [ ] C1 passes (target well inside the top decile of vocabulary, not at chance)
- [ ] C4 passes (in-window ceiling beats every evicted condition)
- [ ] C2 recorded — if the ratio is under 10×, RQ2's claim weakens to "semantic gist"
- [ ] C3 recorded — **if it fails, message Gautam before doing anything else**
- [ ] `04_retention_rows.json` saved

Analysis and figures are in **05_analysis_and_figures.ipynb**, which runs on CPU. Download
`04_retention_rows.json` and run 05 on your laptop — GPU time is the scarce resource,
analysis time is not.
